# Exploratory Data Analysis — Raman Spectroscopy Consensus Spectra

Dataset: `combined_raman_spectra.xlsx`  
Samples: Milk spectra from Amul, Heritage, Nandani, and Nestle brands across varying conditions (temperature, pH, laser wavelength).

**Preprocessing pipeline applied before EDA:**
1. Savitzky-Golay smoothing (window = 5, polynomial order = 2)
2. Asymmetric Least Squares (ALS) baseline correction
3. L² (Euclidean) normalisation

> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads `combined_raman_spectra.xlsx`, compiled from those spectra; the processed authentic dataset is provided (aligned) as `data/raman_valid_samples_combined_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.signal import find_peaks, savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

## 1. Load & Parse Data

In [ ]:
df = pd.read_excel('combined_raman_spectra.xlsx')
raman_shift = df['Raman Shift'].values

# Drop the two Heritage spectra acquired at non-standard laser wavelengths (633 nm & 785 nm)
exclude = [c for c in df.columns if '633nm' in c or '785nm' in c]
df = df.drop(columns=exclude)
print(f'Dropped {len(exclude)} spectra with non-standard laser wavelengths: {exclude}')

spectra_raw = df.drop(columns=['Raman Shift'])

print(f'\nRaman shift range : {raman_shift.min():.1f} – {raman_shift.max():.1f} cm⁻¹')
print(f'Data points       : {len(raman_shift)}')
print(f'Number of spectra : {spectra_raw.shape[1]}')
print(f'Missing values    : {spectra_raw.isna().sum().sum()}')

In [ ]:
def parse_metadata(col):
    col_clean = col.replace('.txt', '')
    brand = col_clean.split()[0]
    milk_type = ('full cream' if 'full cream' in col_clean.lower() else
                 'toned'      if 'toned'      in col_clean.lower() else
                 'special'    if 'special'    in col_clean.lower() else 'unknown')
    if 'after 24' in col_clean.lower():
        condition = 'after_24h'
    elif 'after heating' in col_clean.lower() or 'heating' in col_clean.lower():
        condition = 'heated'
    elif 'cold'   in col_clean.lower(): condition = 'cold'
    elif 'basic'  in col_clean.lower(): condition = 'basic_pH'
    elif 'acidic' in col_clean.lower(): condition = 'acidic_pH'
    else: condition = 'RT'
    laser = '532nm'
    for wl in ['532nm', '633nm', '785nm']:
        if wl in col_clean:
            laser = wl
            break
    return pd.Series({'brand': brand, 'milk_type': milk_type, 'condition': condition, 'laser': laser})

meta = spectra_raw.columns.to_series().apply(parse_metadata)
meta.index = spectra_raw.columns
print(meta['brand'].value_counts(), '\n')
print(meta['milk_type'].value_counts(), '\n')
print(meta['condition'].value_counts(), '\n')
print(meta['laser'].value_counts())

## 2. Preprocessing Pipeline

In [ ]:
# ── Step 1: Savitzky-Golay smoothing ──────────────────────────────────────────
def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

# ── Step 2: ALS baseline correction (Eilers & Boelens 2005) ───────────────────
def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

# ── Step 3: L² normalisation ──────────────────────────────────────────────────
def l2_normalize(y):
    norm = np.linalg.norm(y)
    return y / norm if norm > 0 else y

# ── Apply pipeline ────────────────────────────────────────────────────────────
preprocessed = {}
for col in spectra_raw.columns:
    y = spectra_raw[col].values.astype(float)
    y = sg_smooth(y)              # 1. Smooth
    y = y - als_baseline(y)       # 2. Subtract ALS baseline
    y = np.clip(y, 0, None)       # keep non-negative
    y = l2_normalize(y)           # 3. L² normalise
    preprocessed[col] = y

spectra = pd.DataFrame(preprocessed, index=df.index)
print('Preprocessing complete.')
print(f'Spectra shape: {spectra.shape}  |  Any NaN: {spectra.isna().any().any()}')

## 3. Dataset Summary (Post-Preprocessing)

In [ ]:
summary = pd.DataFrame({
    'Mean intensity': spectra.mean(),
    'Std intensity' : spectra.std(),
    'Min intensity' : spectra.min(),
    'Max intensity' : spectra.max(),
    'SNR (mean/std)': spectra.mean() / spectra.std()
}).join(meta)

summary[['brand','milk_type','condition','laser','Mean intensity','Std intensity','SNR (mean/std)']].round(4)

## 4. All Spectra Overview

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = cm.tab10(np.linspace(0, 1, spectra.shape[1]))
for i, col in enumerate(spectra.columns):
    ax.plot(raman_shift, spectra[col], alpha=0.5, linewidth=0.7, color=colors[i])
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('All Spectra Overlay (Preprocessed)')
plt.tight_layout()
plt.show()

## 5. Spectra by Brand

In [ ]:
brands = meta['brand'].unique()
fig, axes = plt.subplots(len(brands), 1, figsize=(14, 4 * len(brands)), sharex=True)

for ax, brand in zip(axes, brands):
    cols = meta[meta['brand'] == brand].index.tolist()
    brand_colors = cm.tab10(np.linspace(0, 1, len(cols)))
    for i, col in enumerate(cols):
        label = col.replace('.txt', '').replace(f'{brand} ', '')
        ax.plot(raman_shift, spectra[col], alpha=0.7, linewidth=0.8,
                color=brand_colors[i], label=label)
    ax.set_title(f'{brand} (n={len(cols)})')
    ax.set_ylabel('Normalised Intensity (a.u.)')
    ax.legend(fontsize=6, ncol=2, loc='upper right')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle('Spectra by Brand (Preprocessed)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Mean Spectrum per Brand

In [ ]:
brand_palette = {'Amul': '#e41a1c', 'Heritage': '#377eb8', 'Nandani': '#4daf4a', 'Nestle': '#984ea3'}

fig, ax = plt.subplots(figsize=(14, 5))
for brand in brands:
    cols = meta[meta['brand'] == brand].index.tolist()
    mean_spec = spectra[cols].mean(axis=1)
    std_spec  = spectra[cols].std(axis=1)
    color = brand_palette.get(brand, 'grey')
    ax.plot(raman_shift, mean_spec, label=brand, color=color, linewidth=1.2)
    ax.fill_between(raman_shift, mean_spec - std_spec, mean_spec + std_spec,
                    color=color, alpha=0.15)

ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Mean ± 1 SD Spectrum per Brand (Preprocessed)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Effect of Measurement Condition (532 nm, all brands)

In [ ]:
conditions = meta['condition'].unique()
cond_palette = dict(zip(conditions, cm.Set2(np.linspace(0, 1, len(conditions)))))

fig, ax = plt.subplots(figsize=(14, 5))
for cond in conditions:
    cols = meta[(meta['condition'] == cond) & (meta['laser'] == '532nm')].index.tolist()
    if not cols:
        continue
    mean_spec = spectra[cols].mean(axis=1)
    ax.plot(raman_shift, mean_spec, label=cond, color=cond_palette[cond], linewidth=1.2)

ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Mean Spectrum by Measurement Condition — 532 nm (Preprocessed)')
ax.legend(title='Condition')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for mtype, color in [('full cream', '#e41a1c'), ('toned', '#377eb8')]:
    cols = meta[(meta['milk_type'] == mtype) & (meta['laser'] == '532nm')].index.tolist()
    if not cols:
        continue
    mean_spec = spectra[cols].mean(axis=1)
    std_spec  = spectra[cols].std(axis=1)
    ax.plot(raman_shift, mean_spec, label=mtype, color=color, linewidth=1.4)
    ax.fill_between(raman_shift, mean_spec - std_spec, mean_spec + std_spec, color=color, alpha=0.15)

ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Full Cream vs. Toned Milk — Mean ± 1 SD, 532 nm (Preprocessed)')
ax.legend(title='Milk type')
plt.tight_layout()
plt.show()

## 10. Intensity Distribution (Boxplot per Brand)

In [ ]:
intensity_df = spectra.T.copy()
intensity_df.columns = [f'pt{i}' for i in range(len(raman_shift))]
intensity_df = intensity_df.join(meta)
melted = intensity_df[['brand'] + [c for c in intensity_df.columns if c.startswith('pt')]].melt(
    id_vars='brand', var_name='point', value_name='intensity')

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=melted, x='brand', y='intensity', palette='Set2', ax=ax,
            flierprops=dict(marker='.', markersize=1, alpha=0.3))
ax.set_xlabel('Brand')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Intensity Distribution per Brand (Preprocessed)')
plt.tight_layout()
plt.show()

## 11. Peak Detection on Grand Mean Spectrum

In [ ]:
grand_mean = spectra.mean(axis=1).values
peaks, _ = find_peaks(grand_mean, prominence=0.001, distance=10)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(raman_shift, grand_mean, linewidth=1, color='steelblue', label='Grand mean')
ax.scatter(raman_shift[peaks], grand_mean[peaks], color='red', zorder=5, s=30, label=f'{len(peaks)} peaks')
for p in peaks:
    ax.annotate(f'{raman_shift[p]:.0f}', (raman_shift[p], grand_mean[p]),
                textcoords='offset points', xytext=(0, 6), fontsize=6, ha='center', rotation=90)
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Peak Detection on Grand Mean Spectrum (Preprocessed)')
ax.legend()
plt.tight_layout()
plt.show()

peak_table = pd.DataFrame({'Raman Shift (cm⁻¹)': raman_shift[peaks].round(1),
                            'Norm. Intensity': grand_mean[peaks].round(6)})
peak_table.sort_values('Norm. Intensity', ascending=False).reset_index(drop=True).head(20)

| Rank | Raman Shift (cm⁻¹) | Norm. Intensity | Category | Chemical Assignment |
|------|---------------------|-----------------|----------|---------------------|
| 1 | 1438.5 | 0.1002 | Lipid | CH₂ scissoring / δ(CH₂) deformation — dominant saturated fatty acid methylene bending band |
| 2 | 2851.7 | 0.0880 | Lipid | Symmetric CH₂ stretching (ν_s CH₂) — fatty acid alkyl chains |
| 3 | 2891.2 | 0.0806 | Lipid | Asymmetric CH₂ stretching (ν_as CH₂) — fatty acid chains; lactose CH₂ minor contribution |
| 4 | 2929.1 | 0.0746 | Lipid | Asymmetric CH₃ stretching (ν_as CH₃) / Fermi resonance component of CH₂ stretch |
| 5 | 1521.4 | 0.0622 | Lipid | Carotenoid C=C conjugated stretch (β-carotene); present in bovine milk fat globules |
| 6 | 1155.7 | 0.0602 | Lactose | C–O bond stretching of polysaccharides — primary lactose marker in milk Raman spectra |
| 7 | 1300.7 | 0.0546 | Lipid | In-phase CH₂ twisting (τCH₂) — saturated fatty acid chains; ubiquitous methylene twist |
| 8 | 1652.5 | 0.0509 | Lipid + Protein | Amide I (protein C=O stretch, α-helix) overlapping with C=C stretch of unsaturated fatty acids |
| 9 | 1899.7 | 0.0303 | Combination | Overtone / combination band in the Raman silent region (~2× ~950 cm⁻¹); unusually high intensity — origin unclear |
| 10 | 1001.5 | 0.0267 | Protein | Phenylalanine symmetric ring breathing mode — unambiguous protein marker (casein/whey) |
| 11 | 1079.0 | 0.0263 | Lipid | C–C stretching of –(CH₂)ₙ– fatty acid chain; lactose C–O vibrations (~1097 cm⁻¹) may overlap |
| 12 | 874.8 | 0.0256 | Lipid | C–C stretching of –(CH₂)ₙ– chain; also tryptophan ring / monosaccharide C–O–C contributions reported |
| 13 | 1062.6 | 0.0246 | Lipid | Skeletal C–C stretch of all-trans fatty acid chain; sensitive to chain order and packing |
| 14 | 1265.5 | 0.0236 | Lipid | In-plane δ(=C–H) deformation of cis double bonds (unsaturated FA); Amide III overlap possible |
| 15 | 3171.8 | 0.0192 | Water | O–H stretching — low-frequency shoulder of the broad water band (strongly H-bonded O–H) |
| 16 | 1743.8 | 0.0191 | Lipid | ν(C=O) ester carbonyl stretch — triglycerides and free fatty acids |
| 17 | 1123.1 | 0.0188 | Lipid | C–C / C–O–C stretching in glycerol ester backbone of triglycerides |
| 18 | 2721.3 | 0.0133 | Combination | First overtone of CH₂ scissoring (~2 × 1438 cm⁻¹) via Fermi resonance with CH₂ symmetric stretch |
| 19 | 2173.5 | 0.0117 | Combination | Weak combination band in the Raman silent region; origin not conclusively assigned in milk literature |
| 20 | 605.2 | 0.0107 | Lipid | C=O in-plane deformation / glycerol C–O–C skeletal bending in triglycerides |

## 12. Correlation Heatmap Between Spectra

In [ ]:
corr = spectra.corr()
short_labels = [
    meta.loc[c, 'brand'] + '_' + meta.loc[c, 'condition'] + '_' + meta.loc[c, 'milk_type'].replace(' ', '')
    for c in corr.columns
]

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr.values, ax=ax, cmap='RdYlGn', vmin=0.8, vmax=1.0,
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.3, linecolor='lightgrey')
ax.set_title('Pearson Correlation Between All Spectra (Preprocessed)', fontsize=13)
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.show()

## 13. PCA — Spectral Variance Structure

In [ ]:
X = spectra.T.values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

print('Explained variance ratio:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v*100:.1f}%')

In [ ]:
brand_colors_map = {'Amul': '#e41a1c', 'Heritage': '#377eb8', 'Nandani': '#4daf4a', 'Nestle': '#984ea3'}
mtype_markers    = {'full cream': 'o', 'toned': 's', 'special': '^', 'unknown': 'D'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (pc_x, pc_y) in zip(axes, [(0, 1), (0, 2)]):
    for i, col in enumerate(spectra.columns):
        brand = meta.loc[col, 'brand']
        mtype = meta.loc[col, 'milk_type']
        ax.scatter(scores[i, pc_x], scores[i, pc_y],
                   color=brand_colors_map.get(brand, 'grey'),
                   marker=mtype_markers.get(mtype, 'o'),
                   s=60, edgecolors='k', linewidths=0.4, alpha=0.85)
    ax.set_xlabel(f'PC{pc_x+1} ({pca.explained_variance_ratio_[pc_x]*100:.1f}%)')
    ax.set_ylabel(f'PC{pc_y+1} ({pca.explained_variance_ratio_[pc_y]*100:.1f}%)')
    ax.set_title(f'PCA Score Plot — PC{pc_x+1} vs PC{pc_y+1} (Preprocessed)')
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.axvline(0, color='grey', lw=0.5, ls='--')

brand_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                         markersize=8, markeredgecolor='k', label=b)
                 for b, c in brand_colors_map.items()]
mtype_handles = [Line2D([0], [0], marker=m, color='grey', markersize=8, label=t)
                 for t, m in mtype_markers.items() if t != 'unknown']
axes[1].legend(handles=brand_handles + mtype_handles, fontsize=8,
               title='Brand / Milk type', loc='lower right')
plt.tight_layout()
plt.show()

## 14. PCA Loadings — Which Wavenumbers Drive Variance?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(raman_shift, pca.components_[i], linewidth=0.9, color=f'C{i}')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_ylabel(f'PC{i+1} loading')
    ax.set_title(f'PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}% variance)')
axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle('PCA Loadings (Preprocessed)', fontsize=13)
plt.tight_layout()
plt.show()

## 15. Signal-to-Noise Ratio Comparison

In [ ]:
snr_df = summary[['brand', 'milk_type', 'condition', 'laser', 'SNR (mean/std)']].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=snr_df, x='brand', y='SNR (mean/std)', hue='milk_type',
            palette='Set2', ax=ax, capsize=0.1, errwidth=1.2)
ax.set_xlabel('Brand')
ax.set_ylabel('SNR (mean / std)')
ax.set_title('Signal-to-Noise Ratio by Brand and Milk Type (Preprocessed)')
ax.legend(title='Milk type')
plt.tight_layout()
plt.show()

## 16. Key Raman Bands — Regional Zoom

In [ ]:
regions = [
    ('C–H deformation / lipid', 1400, 1500),
    ('Amide I (protein C=O)',    1600, 1700),
    ('C–H stretch (lipid/protein)', 2800, 3050),
    ('O–H stretch (water)',      3100, 3500),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
for ax, (title, lo, hi) in zip(axes, regions):
    mask = (raman_shift >= lo) & (raman_shift <= hi)
    for brand in brands:
        cols = meta[(meta['brand'] == brand) & (meta['laser'] == '532nm')].index.tolist()
        if not cols:
            continue
        mean_spec = spectra[cols].mean(axis=1)
        ax.plot(raman_shift[mask], mean_spec[mask], label=brand,
                color=brand_palette.get(brand, 'grey'), linewidth=1.2)
    ax.set_title(f'{title}\n({lo}–{hi} cm⁻¹)')
    ax.set_xlabel('Raman Shift (cm⁻¹)')
    ax.set_ylabel('Normalised Intensity (a.u.)')
    ax.legend(fontsize=8)

plt.suptitle('Key Raman Spectral Regions (Preprocessed)', fontsize=13)
plt.tight_layout()
plt.show()

## EDA Summary: Feasibility of a Consensus Raman Spectrum for Milk Quality Control 

The primary goal of this EDA was not to discriminate between brands or milk types, but to assess whether a representative consensus spectrum can be constructed from the 44 spectra collected across varying brands, milk types, and physicochemical conditions. The findings broadly support this feasibility.

Shared spectral fingerprint across samples. After preprocessing (Savitzky-Golay smoothing, ALS baseline correction, and L² normalisation), all 44 spectra show a consistent underlying milk fingerprint. The dominant peaks on the grand mean spectrum —the C–H deformation band at ~1438 cm⁻¹, the C–H stretch cluster at ~2851, 2891, and 2929 cm⁻¹, and the Amide I band at ~1652 cm⁻¹ — appear reliably across all samples regardless of brand or condition. This reproducibility of the core spectral structure is a necessary precondition for constructing a meaningful consensus, and the EDA confirms it holds.

PCA variance structure supports consensus construction. PCA on the preprocessed spectra revealed that PC1 accounts for 37.8% of variance and PC2 for 23.1%, with PC3 contributing a further 9.5% — together capturing approximately 70% of total spectral variance across just three components. Critically, the PC1 vs PC2 score plot shows that the majority of samples cluster reasonably tightly in spectral space, with only a handful of samples sitting at larger distances from the main group. This degree of clustering, even across diverse brands, milk types, and conditions, indicates that the dataset is governed by a common latent structure — the biochemical fingerprint of uncontaminated milk — rather than by fragmented, sample-specific variation. This is precisely what one would want to see before computing a consensus: the spectra are sufficiently concordant that an average or barycentric representation will be informative rather than an artefact of averaging over incompatible signals.

Residual variance in PC3 and beyond is driven by water and fat content. The remaining variance, captured particularly in PC3 (9.5%), is not random noise but has a clear physical origin. The PC3 loadings show concentrated, localised contributions in the C–H stretch region (~2850–2930 cm⁻¹) and the O–H stretch region (~3100–3500 cm⁻¹), corresponding to fat and water signals respectively. These are the two compositional variables most likely to differ across brands (fat percentage) and across measurement conditions such as temperature and pH (which alter the hydrogen-bonding environment of water). The regional zoom plots confirm this directly — the C–H stretch and O–H stretch bands show the most inter-sample spread, while the fingerprint region (400–1800 cm⁻¹) remains comparatively stable. This is entirely expected and does not undermine the feasibility of a consensus spectrum; it simply means that these two regions will carry the most uncertainty in any aggregated representation, which should be accounted for in the consensus construction methodology (e.g. through robust median or Wasserstein barycenter approaches rather than a simple arithmetic mean).

Other observations. The effect of measurement condition (RT, cold, heated, after 24h, basic pH) on the mean spectrum is minor at the whole-spectrum level, which is encouraging — it suggests that physicochemical variation across the collection conditions does not introduce systematic spectral distortions that would corrupt a consensus. SNR varies somewhat across brands and milk types, with toned milk samples generally showing slightly higher SNR, though this is a data quality observation rather than a structural concern. The 785 nm Heritage spectrum shows elevated fluorescence background, which is effectively removed by the ALS baseline correction.

Taken together, the EDA establishes that the dataset has the structural properties needed to support consensus spectrum construction: a stable, reproducible core fingerprint across all samples, tight clustering in PCA space, and residual variance that is physically interpretable and concentrated in predictable spectral regions.